# Ly-alpha forest correlations across cTreeBalls engines

One retained NumPy catalog, seven OpenMP engines and seven MPI counterparts. Run the small synthetic example first. Radial and anisotropic 3D statistics have different selection rules and are compared only within their own families.

Use the rebuilt cyballs environment with NumPy, SciPy, Astropy and Matplotlib. See `python/README_lya_corr_all_engines.md` for configuration and scientific limitations.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'python/lya_corr_all_engines.py').is_file())
sys.path.insert(0, str(root / 'python'))
from lya_corr_all_engines import (read_desi, synthetic_catalog, save_catalog,
    discover_cython_methods, LYA_ENGINES, resolve_engines, RunConfig, run_engine_suite)

available = discover_cython_methods(list(LYA_ENGINES))
print('\n'.join(available))

## Select input

Synthetic input needs no download. To try real data, run the following command from the project root, then set `USE_DESI = True`:

```bash
bash examples/download_desi_lya_example.sh data/desi-lya-example
```

This downloads one small DR1 file, not the full release. DESI `LOS_ID` is kept as int64. The reader preserves `DELTA_BLIND` and `WEIGHT`, records the blinding label, and converts pixel wavelength to comoving distance. The subset and stride below are only for testing; they are not survey cuts or rebinning.

In [ ]:
USE_DESI = False
if USE_DESI:
    catalog = read_desi([str(root / 'data/desi-lya-example/delta-1019.fits.gz')],
                        max_forests=6, pixel_stride=30, omega_m=0.315, h=0.674)
else:
    catalog = synthetic_catalog(forests=8, pixels=12)
print(catalog.nbody, 'pixels')
print(json.dumps(catalog.metadata, indent=2))

## Run OpenMP methods

The seven methods cover 3D 2PCF, 5D 3PCF, radial 2PCF (scan and interval tree), radial 3PCF, and combined runs. All use the same input arrays; trees are rebuilt per engine. `statistics='both'` opts into potentially expensive 3PCF. For a large input, start with `statistics='2pcf'`.

In [ ]:
engines = resolve_engines(['all-omp'], available, statistics='both')
output = root / 'Output_lya_notebook' / datetime.now().strftime('%Y%m%d-%H%M%S-%f')
config = RunConfig(engines, output, threads=2, rp_bins=8, rt_bins=8, r3_max=60)
results = run_engine_suite(catalog, config)
summary = json.loads((output / 'summary.json').read_text())
print('Results:', output)
print(json.dumps(summary['engines'], indent=2))

## Compare normalized correlations

These figures are kept separate by estimator. The 5D 3PCF image sums raw angular numerators and denominators at fixed (r1,r2), then divides. Empty bins are blank. CSV files include full-bin absolute and relative differences between methods measuring the same statistic; relative differences at zero reference values are undefined (`nan`).

In [ ]:
from IPython.display import Image, display
for path in summary['plots']:
    display(Image(filename=path))
print(json.dumps(summary['comparisons'], indent=2))

## Optional MPI run

MPI should run in separate Python processes launched from the notebook, not just on one kernel rank. Enable the cell below only when mpi4py and cyballs use the same MPI. It reuses a saved NPZ catalog; rank zero loads and broadcasts it once. On a cluster, use the site's MPI launcher and a shared output directory. The direct Python API also accepts your own `ForestCatalog(positions, delta, weights, forest_ids)`.

In [ ]:
RUN_MPI = False
if RUN_MPI:
    import subprocess
    cache = output / 'pixels.npz'
    save_catalog(cache, catalog)
    command = [sys.executable, str(root / 'python/lya_corr_all_engines.py'),
               '--catalog', str(cache), '--engine', 'all', '--statistics', 'both',
               '--mpi-ranks', '2', '--threads', '2', '--r3-max', '60',
               '--output', str(output / 'mpi')]
    subprocess.run(command, cwd=root, check=True)